In [4]:
import re
import zipfile
from pathlib import Path

SONGS_DIR = Path(r"C:\Users\Justin\AppData\Local\osu!\Songs")  # Update this path
OUT_DIR = Path("data")
OUT_ZIP = OUT_DIR / "osu_50.zip"
MAX_SONGS = 50

In [5]:
def parse_header(osu_path: Path):
    text = osu_path.read_text(encoding="utf-8", errors="ignore")
    audio = re.search(r"^AudioFilename:\s*(.+)$", text, re.MULTILINE)
    mode = re.search(r"^Mode:\s*(\d+)", text, re.MULTILINE)
    audio_name = audio.group(1).strip() if audio else None
    mode_val = int(mode.group(1)) if mode else 0  # Older formats default to 0
    return audio_name, mode_val

In [6]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
count = 0
with zipfile.ZipFile(OUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in sorted(SONGS_DIR.iterdir()):
        if not folder.is_dir():
            continue

        osu_files = []
        audio_files = set()

        for osu_file in folder.glob("*.osu"):
            audio_name, mode = parse_header(osu_file)
            if mode != 0 or not audio_name:  # Keep osu!standard only
                continue

            audio_path = folder / audio_name
            if not audio_path.exists():
                continue

            osu_files.append(osu_file)
            audio_files.add(audio_path)
            count += 1
            if count >= MAX_SONGS:
                break

        if not osu_files:
            continue

        # Write only the .osu maps and their referenced audio into the zip
        for f in osu_files + list(audio_files):
            zf.write(f, arcname=f"{folder.name}/{f.name}")

        if count >= MAX_SONGS:
            break

print(f"Packed {count} .osu files into {OUT_ZIP} ({OUT_ZIP.stat().st_size / 1e6:.1f} MB)")

Packed 50 .osu files into data\osu_50.zip (60.1 MB)
